# IETF Ambiguity Detection — Classical ML Experiments

This notebook records the feature cleanup, model comparisons, threshold selection, and context experiments. The existing train/validation/test assignments are preserved. Results on the test family are exploratory because they have been inspected during development.

**Data:** 2,546 labeled clauses across 11 families (455 ambiguous; 17.9%). **Evaluation:** AP (average precision), precision, recall, F1, and confusion matrices. The family-grouped results leave a whole family out; all learned text preprocessing is fitted on each fold's training rows only. No annotation evidence or future-revision information is used as a feature.

## Contents
1. Data preparation and numeric baseline
2. Feature representations on the existing split
3. Validation-only threshold selection
4. Family-held-out model comparison
5. Previous-clause context experiment
6. Family-size sensitivity and earlier revision experiment

In [1]:
import pandas as pd
import numpy as np

In [2]:
# ============================================================
# Feature cleanup for ambiguity detection dataset
# Goal:
# 1) remove leakage / metadata columns
# 2) remove redundant binary markers
# 3) keep the core predictive signal
# 4) create aggregate features for model stability
# ============================================================
df=pd.read_csv('/media/yahya/data_stuff/DATA/Final Project/ietf-ambiguity-detection/corpus/processed/features/v3/audit/engineered_features.csv')
df_t_labels=pd.read_csv('/media/yahya/data_stuff/DATA/Final Project/ietf-ambiguity-detection/corpus/processed/features/v3/targets/train_labels.csv')
# Safe copy of the working dataframe
df_model = df.copy()

# ------------------------------------------------------------
# 1) Drop leakage columns
# These are not model features. They may leak document identity
# or dataset assignment into the training process.
# ------------------------------------------------------------
leakage_cols = [
    "record_id",
    "split",
    "family_id"
]

for col in leakage_cols:
    if col in df_model.columns:
        df_model = df_model.drop(columns=[col])

# ------------------------------------------------------------
# 2) Drop redundant binary presence flags
# If the count is already present, the binary `*_present` flag
# is usually repetitive and does not add much signal.
# ------------------------------------------------------------
redundant_present_cols = [
    "vague_term_present",
    "anaphora_present",
    "cross_reference_present",
    "conditional_present",
    "exception_present",
    "temporal_present",
    "ordering_present",
    "normative_bcp14_present",
    "context_previous_present",
    "context_next_present"
]

for col in redundant_present_cols:
    if col in df_model.columns:
        df_model = df_model.drop(columns=[col])

# ------------------------------------------------------------
# 3) Remove count/density duplicates
# Keep either count or density, not both. Count-based patterns
# are usually easier to interpret and often more robust.
# ------------------------------------------------------------
density_cols = [
    "vague_term_density",
    "anaphora_density",
    "cross_reference_density",
    "conditional_density",
    "exception_density",
    "temporal_density",
    "ordering_density"
]

for col in density_cols:
    if col in df_model.columns:
        df_model = df_model.drop(columns=[col])

# ------------------------------------------------------------
# 4) Remove redundant derived averages
# These are often strongly correlated with raw counts and can
# inflate multicollinearity without adding much unique signal.
# ------------------------------------------------------------
derived_avg_cols = [
    "basic_average_word_length",
    "basic_average_sentence_words",
    "basic_average_sentence_characters"
]

for col in derived_avg_cols:
    if col in df_model.columns:
        df_model = df_model.drop(columns=[col])

# ------------------------------------------------------------
# 5) Retain the strongest feature set
# These are the main lexical, discourse, normative, and syntax
# signals relevant to ambiguity detection in standards text.
# ------------------------------------------------------------
keep_cols = [
    "basic_word_count",
    "basic_character_count",
    "basic_sentence_count",
    "vague_term_count",
    "anaphora_count",
    "cross_reference_count",
    "conditional_count",
    "exception_count",
    "temporal_count",
    "ordering_count",
    "normative_bcp14_count",
    "normative_bcp14_unique_count",
    "normative_obligation_count",
    "normative_recommendation_count",
    "normative_permission_count",
    "normative_not_recommended_count",
    "normative_should_not_count",
    "normative_shall_not_count",
    "normative_must_not_count",
    "normative_recommended_count",
    "normative_required_count",
    "normative_optional_count",
    "normative_should_count",
    "normative_shall_count",
    "normative_must_count",
    "normative_may_count",
    "syntax_mean_dependency_distance",
    "syntax_max_dependency_depth",
    "syntax_subordinate_clause_count",
    "syntax_coordinate_conjunction_count",
    "context_previous_word_count",
    "context_previous_character_count",
    "context_previous_sentence_count",
    "context_next_word_count",
    "context_next_character_count",
    "context_next_sentence_count",
    "context_section_depth",
    "label"
]

available_keep_cols = [c for c in keep_cols if c in df_model.columns]
df_model = df_model[available_keep_cols]

# ------------------------------------------------------------
# 6) Build compact aggregate features
# These reduce noise and keep the model from overfitting to
# many nearly duplicated subcomponents.
# ------------------------------------------------------------

normative_cols = [
    "normative_obligation_count",
    "normative_recommendation_count",
    "normative_permission_count",
    "normative_not_recommended_count",
    "normative_should_not_count",
    "normative_shall_not_count",
    "normative_must_not_count",
    "normative_recommended_count",
    "normative_required_count",
    "normative_optional_count",
    "normative_should_count",
    "normative_shall_count",
    "normative_must_count",
    "normative_may_count"
]

normative_present = [c for c in normative_cols if c in df_model.columns]
if normative_present:
    df_model["total_normative_count"] = df_model[normative_present].sum(axis=1)

ambiguity_cols = [
    "vague_term_count",
    "anaphora_count",
    "cross_reference_count",
    "conditional_count",
    "exception_count",
    "temporal_count",
    "ordering_count"
]

ambiguity_present = [c for c in ambiguity_cols if c in df_model.columns]
if ambiguity_present:
    df_model["total_ambiguity_signal"] = df_model[ambiguity_present].sum(axis=1)

syntax_cols = [
    "syntax_max_dependency_depth",
    "syntax_subordinate_clause_count",
    "syntax_coordinate_conjunction_count"
]

syntax_present = [c for c in syntax_cols if c in df_model.columns]
if syntax_present:
    df_model["syntax_complexity_score"] = df_model[syntax_present].sum(axis=1)

# ------------------------------------------------------------
# 7) Final sanity check
# Review the cleaned dataframe before model fitting.
# ------------------------------------------------------------
print("Cleaned dataset shape:", df_model.shape)
print("Feature columns:")
for c in df_model.columns:
    print("- ", c)
print("\nPreview:")
display(df_model.head())


Cleaned dataset shape: (2546, 40)
Feature columns:
-  basic_word_count
-  basic_character_count
-  basic_sentence_count
-  vague_term_count
-  anaphora_count
-  cross_reference_count
-  conditional_count
-  exception_count
-  temporal_count
-  ordering_count
-  normative_bcp14_count
-  normative_bcp14_unique_count
-  normative_obligation_count
-  normative_recommendation_count
-  normative_permission_count
-  normative_not_recommended_count
-  normative_should_not_count
-  normative_shall_not_count
-  normative_must_not_count
-  normative_recommended_count
-  normative_required_count
-  normative_optional_count
-  normative_should_count
-  normative_shall_count
-  normative_must_count
-  normative_may_count
-  syntax_mean_dependency_distance
-  syntax_max_dependency_depth
-  syntax_subordinate_clause_count
-  syntax_coordinate_conjunction_count
-  context_previous_word_count
-  context_previous_character_count
-  context_previous_sentence_count
-  context_next_word_count
-  context_nex

,basic_word_count,basic_character_count,basic_sentence_count,vague_term_count,anaphora_count,cross_reference_count,conditional_count,exception_count,temporal_count,ordering_count,...,context_previous_word_count,context_previous_character_count,context_previous_sentence_count,context_next_word_count,context_next_character_count,context_next_sentence_count,context_section_depth,total_normative_count,total_ambiguity_signal,syntax_complexity_score
0,36,241,1,0,2,0,2,0,0,0,...,13,84,1,15,103,1,1,2,4,0
1,43,242,1,0,3,0,0,0,0,1,...,5,38,1,0,0,0,2,2,4,0
2,39,254,1,0,0,1,0,0,0,0,...,0,0,0,10,53,1,2,2,1,0
3,30,173,1,0,1,0,1,0,0,0,...,37,248,1,16,93,1,2,2,2,0
4,15,83,1,0,1,0,1,0,0,0,...,18,106,1,17,101,1,2,2,2,0


### Feature cleanup notes

The numeric baseline removes IDs, family/split metadata, redundant presence flags and density duplicates. The v3 artifacts already contain aligned engineered, TF-IDF, and combined matrices, with preprocessing fitted on training data. The notebook keeps those saved artifacts unchanged.

## 1. Data preparation and numeric baseline

In [3]:
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Merge saved labels while retaining the existing family-level split.
labels_dir = Path("/media/yahya/data_stuff/DATA/Final Project/ietf-ambiguity-detection/corpus/processed/features/v3/targets")
labels = pd.concat([pd.read_csv(labels_dir / f"{split}_labels.csv") for split in ("train", "validation", "test")], ignore_index=True)
model_data = df_model.drop(columns=["label"], errors="ignore").copy()
model_data.insert(0, "record_id", df["record_id"].to_numpy())
model_data.insert(1, "split", df["split"].to_numpy())
model_data = model_data.merge(labels, on="record_id", how="inner", validate="one_to_one")
assert model_data["label"].notna().all()
assert set(model_data["label"].unique()) == {0, 1}

feature_cols = [c for c in df_model.columns if c != "label"]
train = model_data[model_data["split"] == "train"]
test = model_data[model_data["split"] == "test"]
X_train, y_train = train[feature_cols], train["label"]
X_test, y_test = test[feature_cols], test["label"]

model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

print(f"Merged rows: {len(model_data)} (train={len(train)}, test={len(test)})")
print(f"Test PR-AUC: {average_precision_score(y_test, y_score):.3f}")
print("Confusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))
print(classification_report(y_test, y_pred, labels=[0, 1], target_names=["not ambiguous", "ambiguous"], zero_division=0))

Merged rows: 2546 (train=2253, test=154)
Test PR-AUC: 0.266
Confusion matrix [ [TN, FP], [FN, TP] ]:
[[56 64]
 [12 22]]
               precision    recall  f1-score   support

not ambiguous       0.82      0.47      0.60       120
    ambiguous       0.26      0.65      0.37        34

     accuracy                           0.51       154
    macro avg       0.54      0.56      0.48       154
 weighted avg       0.70      0.51      0.55       154



## 2. Feature representations on the existing split

This compares engineered-only, TF-IDF-only, and combined features using the saved train/validation/test split. Thresholds are selected on validation only; test results are exploratory.

In [4]:
# Controlled comparison: engineered-only, TF-IDF-only, and combined features.
# Preserve the saved splits; choose each threshold on validation only.
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

features_dir = Path('/media/yahya/data_stuff/DATA/Final Project/ietf-ambiguity-detection/corpus/processed/features/v3')
split_data = {}
for split_name in ('train', 'validation', 'test'):
    rows = pd.read_csv(features_dir / 'rows' / f'{split_name}_rows.csv', dtype={'record_id': str, 'family_id': str})
    split_labels = pd.read_csv(features_dir / 'targets' / f'{split_name}_labels.csv', dtype={'record_id': str})
    split_data[split_name] = rows.merge(split_labels, on='record_id', how='inner', validate='one_to_one')
    assert len(split_data[split_name]) == len(rows) == len(split_labels)

families_by_split = {name: set(frame['family_id']) for name, frame in split_data.items()}
assert not (families_by_split['train'] & families_by_split['validation'])
assert not (families_by_split['train'] & families_by_split['test'])
assert not (families_by_split['validation'] & families_by_split['test'])
print({name: {'rows': len(frame), 'families': sorted(frame['family_id'].unique())} for name, frame in split_data.items()})

representations = ('engineered', 'tfidf', 'combined')
matrices = {
    representation: {
        split_name: sparse.load_npz(features_dir / 'matrices' / f'{split_name}_{representation}.npz')
        for split_name in ('train', 'validation', 'test')
    }
    for representation in representations
}
for representation, split_matrices in matrices.items():
    for split_name, matrix in split_matrices.items():
        assert matrix.shape[0] == len(split_data[split_name]), (representation, split_name, matrix.shape)

threshold_grid = np.arange(0.05, 1.00, 0.05)
comparison_rows = []
for representation in representations:
    classifier = LogisticRegression(max_iter=2000, class_weight='balanced')
    classifier.fit(matrices[representation]['train'], split_data['train']['label'])
    val_probability = classifier.predict_proba(matrices[representation]['validation'])[:, 1]
    test_probability = classifier.predict_proba(matrices[representation]['test'])[:, 1]
    y_val = split_data['validation']['label'].to_numpy()
    y_test = split_data['test']['label'].to_numpy()

    val_f1 = [precision_recall_fscore_support(y_val, val_probability >= threshold, average='binary', zero_division=0)[2] for threshold in threshold_grid]
    best_index = max(range(len(threshold_grid)), key=lambda index: (val_f1[index], threshold_grid[index]))
    selected_threshold = float(threshold_grid[best_index])
    val_prediction = val_probability >= selected_threshold
    test_prediction = test_probability >= selected_threshold
    val_precision, val_recall, val_f1_score, _ = precision_recall_fscore_support(y_val, val_prediction, average='binary', zero_division=0)
    test_precision, test_recall, test_f1_score, _ = precision_recall_fscore_support(y_test, test_prediction, average='binary', zero_division=0)

    comparison_rows.append({
        'representation': representation,
        'validation_threshold': selected_threshold,
        'validation_AP': average_precision_score(y_val, val_probability),
        'validation_precision': val_precision,
        'validation_recall': val_recall,
        'validation_F1': val_f1_score,
        'test_AP_exploratory': average_precision_score(y_test, test_probability),
        'test_precision_exploratory': test_precision,
        'test_recall_exploratory': test_recall,
        'test_F1_exploratory': test_f1_score,
    })
    print(f'\n{representation}: validation CM={confusion_matrix(y_val, val_prediction, labels=[0, 1]).tolist()}; test CM={confusion_matrix(y_test, test_prediction, labels=[0, 1]).tolist()}')

comparison_results = pd.DataFrame(comparison_rows).round(3)
print(comparison_results.to_string(index=False))

# Numeric-only family CV refits imputation and scaling within each fold.
# The repository has no raw clause-text source, so TF-IDF cannot be refit
# inside family folds for a comparable leakage-safe CV estimate.
train_meta = split_data['train'][['record_id', 'family_id', 'label']]
engineered = pd.read_csv(features_dir / 'audit' / 'engineered_features.csv', dtype={'record_id': str})
engineered_train = engineered.merge(train_meta[['record_id', 'label']], on='record_id', how='inner', validate='one_to_one')
assert len(engineered_train) == len(train_meta)
metadata_columns = {'record_id', 'family_id', 'split', 'label'}
numeric_columns = [column for column in engineered_train.select_dtypes(include=np.number).columns if column not in metadata_columns]
X_grouped = engineered_train[numeric_columns]
y_grouped = engineered_train['label'].to_numpy()
groups = engineered_train['family_id'].to_numpy()
group_cv = GroupKFold(n_splits=min(5, len(np.unique(groups))))
oof_probability = np.full(len(engineered_train), np.nan)
for fold_train, fold_valid in group_cv.split(X_grouped, y_grouped, groups):
    fold_model = make_pipeline(
        SimpleImputer(strategy='median'),
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight='balanced'),
    )
    fold_model.fit(X_grouped.iloc[fold_train], y_grouped[fold_train])
    oof_probability[fold_valid] = fold_model.predict_proba(X_grouped.iloc[fold_valid])[:, 1]
assert np.isfinite(oof_probability).all()
grouped_cv_result = {
    'families': len(np.unique(groups)),
    'folds': group_cv.get_n_splits(),
    'OOF_AP': round(average_precision_score(y_grouped, oof_probability), 3),
    'OOF_F1_at_0.5': round(precision_recall_fscore_support(y_grouped, oof_probability >= 0.5, average='binary', zero_division=0)[2], 3),
}
print('Numeric-only grouped CV on training families:', grouped_cv_result)


{'train': {'rows': 2253, 'families': ['draft-ietf-6man-rfc2460bis', 'draft-ietf-httpbis-cache-header', 'draft-ietf-httpbis-http2bis', 'draft-ietf-httpbis-semantics', 'draft-ietf-quic-http', 'draft-ietf-quic-recovery', 'draft-ietf-quic-tls', 'draft-ietf-quic-transport', 'draft-ietf-tls-tls13']}, 'validation': {'rows': 139, 'families': ['draft-ietf-tcpm-rfc793bis']}, 'test': {'rows': 154, 'families': ['draft-ietf-dnsop-svcb-https']}}

engineered: validation CM=[[105, 8], [18, 8]]; test CM=[[92, 28], [21, 13]]

tfidf: validation CM=[[62, 51], [9, 17]]; test CM=[[73, 47], [13, 21]]

combined: validation CM=[[62, 51], [9, 17]]; test CM=[[59, 61], [10, 24]]
representation  validation_threshold  validation_AP  validation_precision  validation_recall  validation_F1  test_AP_exploratory  test_precision_exploratory  test_recall_exploratory  test_F1_exploratory
    engineered                  0.65          0.290                  0.50              0.308          0.381                0.290         

## 3. Choose an operating threshold on validation

The following cell selects the threshold maximizing validation F1 for the combined logistic-regression model, then reports test performance without using test labels to choose the threshold.

## Threshold experiment for the combined logistic-regression model

The classifier and training data are unchanged. I selected the threshold that maximizes F1 on the existing validation family only, then applied it to the existing test family as an exploratory check. A lower threshold favors recall and creates more false positives. The validation set is only one family (139 records), and the test family has already been examined in previous experiments, so the test result is not an unbiased final estimate. Existing sample weights were not used because their meaning (0.5/1.0) is not documented in a way that justifies changing model fitting.

In [5]:
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_fscore_support

features_dir = Path('/media/yahya/data_stuff/DATA/Final Project/ietf-ambiguity-detection/corpus/processed/features/v3')
splits = ('train', 'validation', 'test')
y_by_split = {
    split: pd.read_csv(features_dir / 'targets' / f'{split}_labels.csv')['label'].astype(int).to_numpy()
    for split in splits
}
X_by_split = {
    split: sparse.load_npz(features_dir / 'matrices' / f'{split}_combined.npz')
    for split in splits
}
assert all(X_by_split[split].shape[0] == len(y_by_split[split]) for split in splits)

threshold_model = LogisticRegression(max_iter=2000, class_weight='balanced')
threshold_model.fit(X_by_split['train'], y_by_split['train'])
validation_probability = threshold_model.predict_proba(X_by_split['validation'])[:, 1]
test_probability = threshold_model.predict_proba(X_by_split['test'])[:, 1]

threshold_candidates = np.unique(validation_probability)
validation_f1 = [
    precision_recall_fscore_support(
        y_by_split['validation'], validation_probability >= threshold,
        average='binary', zero_division=0
    )[2]
    for threshold in threshold_candidates
]
best_validation_threshold = float(max(
    zip(threshold_candidates, validation_f1), key=lambda item: (item[1], item[0])
)[0])

threshold_rows = []
for split, probability in (('validation', validation_probability), ('test (exploratory)', test_probability)):
    target_split = 'test' if split.startswith('test') else split
    for threshold_name, threshold in (('default 0.50', 0.50), ('validation-selected', best_validation_threshold)):
        prediction = probability >= threshold
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_by_split[target_split], prediction, average='binary', zero_division=0
        )
        threshold_rows.append({
            'split': split,
            'threshold': threshold_name,
            'threshold_value': threshold,
            'precision': precision,
            'recall': recall,
            'F1': f1,
            'confusion_matrix_[TN,FP,FN,TP]': confusion_matrix(
                y_by_split[target_split], prediction, labels=[0, 1]
            ).ravel().tolist(),
        })

print(f'Validation-selected threshold: {best_validation_threshold:.3f}')
print(f"Validation AP={average_precision_score(y_by_split['validation'], validation_probability):.3f}; "
      f"test AP (exploratory)={average_precision_score(y_by_split['test'], test_probability):.3f}")
print(pd.DataFrame(threshold_rows).round(3).to_string(index=False))

Validation-selected threshold: 0.422
Validation AP=0.283; test AP (exploratory)=0.315
             split           threshold  threshold_value  precision  recall    F1 confusion_matrix_[TN,FP,FN,TP]
        validation        default 0.50            0.500      0.250   0.385 0.303               [83, 30, 16, 10]
        validation validation-selected            0.422      0.283   0.654 0.395                [70, 43, 9, 17]
test (exploratory)        default 0.50            0.500      0.305   0.529 0.387               [79, 41, 16, 18]
test (exploratory) validation-selected            0.422      0.300   0.706 0.421               [64, 56, 10, 24]


## 4. Family-held-out model comparisons

Leave one document family out at a time. This tests transfer to unseen families and is distinct from the single fixed-split experiment above.

## Full-dataset comparison: leave-one-family-out cross-validation

To use all 2,546 rows without scoring a row with a model trained on that row or on its document family, I ran 11 leave-one-family-out folds. TF-IDF, imputation, and scaling were refit within each fold. Features were clause text and the existing engineered features only; labels, rationales, annotation evidence, and future revisions were not inputs. The fixed 0.50 threshold is descriptive and was not tuned on these out-of-fold predictions.

| Representation | Pooled OOF average precision | Mean family AP (SD) | Precision | Recall | F1 | Confusion matrix [[TN, FP], [FN, TP]] |
|---|---:|---:|---:|---:|---:|---|
| Engineered numerical | 0.275 | 0.299 (0.210) | 0.265 | 0.574 | 0.362 | [[1366, 725], [194, 261]] |
| TF-IDF | 0.273 | 0.320 (0.248) | 0.289 | 0.371 | 0.325 | [[1676, 415], [286, 169]] |
| Combined | **0.293** | **0.331 (0.253)** | **0.301** | 0.470 | **0.367** | [[1593, 498], [241, 214]] |

There are 455 positive rows (17.9%), so the prevalence baseline for average precision is 0.179; the majority-class accuracy baseline is 0.821. By-family AP is engineered / TF-IDF / combined: DNS SVCB 0.302 / 0.297 / 0.309; HTTP/2 0.347 / 0.297 / 0.375; HTTP semantics 0.353 / 0.344 / 0.372; QUIC HTTP 0.172 / 0.207 / 0.193; QUIC recovery 0.353 / 0.358 / 0.404; QUIC TLS 0.111 / 0.143 / 0.118; QUIC transport 0.309 / 0.304 / 0.334; TCP 0.295 / 0.253 / 0.255; TLS 1.3 0.220 / 0.312 / 0.284. The 6man family has only one row (no positive examples), and cache-header has five; their family AP scores are not statistically meaningful. Grouped results vary widely across families, and this small dataset does not support a strong generalization claim. Treat these as exploratory CV results, not a final unbiased benchmark.

## Two additional classifiers: leave-one-family-out cross-validation

The same 11-family leave-one-family-out evaluation was run for two models chosen for complementary reasons: **LinearSVC on TF-IDF** is a strong, efficient high-dimensional sparse-text baseline; **Random Forest on engineered numerical features** can model nonlinear feature interactions. For a like-for-like comparison, each is paired with logistic regression on the same feature representation, with identical held-out families. Class balancing is enabled. Thresholds are the models' defaults (0.5 for logistic regression and Random Forest; 0 for SVC), not validation-tuned.

| Model | Features | Pooled OOF AP | Mean family AP (10 families with positives) | Precision | Recall | F1 | Confusion matrix [[TN, FP], [FN, TP]] |
|---|---|---:|---:|---:|---:|---:|---|
| Logistic Regression | TF-IDF | 0.273 | 0.352 | 0.289 | 0.371 | 0.325 | [[1676, 415], [286, 169]] |
| LinearSVC | TF-IDF | 0.254 | 0.330 | 0.292 | 0.266 | 0.278 | [[1798, 293], [334, 121]] |
| Logistic Regression | Engineered numerical | 0.275 | 0.329 | 0.265 | 0.574 | 0.362 | [[1366, 725], [194, 261]] |
| Random Forest | Engineered numerical | 0.240 | 0.309 | 0.319 | 0.099 | 0.151 | [[1995, 96], [410, 45]] |

Average precision baseline from positive prevalence is 0.179. On matching feature representations and folds, logistic regression scores better than LinearSVC on TF-IDF in pooled AP and F1. Logistic regression also scores better than Random Forest on engineered features in pooled AP, recall, and F1, while the forest has fewer false positives at its default threshold. No threshold or hyperparameter tuning was done. These are exploratory cross-validation results, not a final external benchmark.

### Keep small families in training, not as held-out folds

Families under 30 rows remain in the training data for every fold; only families with at least 30 rows are used for held-out evaluation.

## LOFO evaluation excluding small families only from held-out folds

Interpretation: families with fewer than 30 clauses are **not removed from training**; their records are included in every fold's training set and excluded only from held-out scoring. This yields 9 held-out families and 2,540 evaluated records (453 positives); the 6 records from the 1-row and 5-row families remain in training. The same folds, TF-IDF, engineered features, and logistic-regression settings are used for baseline and previous-clause context. All preprocessing is fitted within each fold. Scores below use the default 0.50 threshold; threshold selection is a separate validation-set task.

| Model | Evaluated AP | Precision | Recall | F1 | Confusion matrix [[TN, FP], [FN, TP]] |
|---|---:|---:|---:|---:|---|
| Baseline (current clause + engineered) | 0.291 | 0.299 | 0.468 | 0.365 | [[1590, 497], [241, 212]] |
| Plus immediately previous clause | 0.292 | 0.300 | 0.355 | 0.325 | [[1711, 376], [292, 161]] |

The context version slightly increases AP (+0.001) but has lower recall and F1 at the default threshold. The test folds are exploratory because the dataset has been examined repeatedly; the two tiny families have no held-out metrics by design.

## 5. Add the immediately previous clause

Use the preceding clause from the same document revision and section as extra TF-IDF input. The target and numeric features stay unchanged.

## Add the immediately preceding clause as text context

This experiment adds TF-IDF features from the preceding clause in the same document revision and section. It does not use a previous revision, labels, annotation evidence, or rationales as features. The target clause and engineered features remain unchanged. The existing train/validation/test assignments are preserved; the context TF-IDF vocabulary is fit on training rows only. The validation set selects an F1 threshold; test results are exploratory.

In [6]:
import json
from scipy import sparse
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_fscore_support

features_dir = Path('/media/yahya/data_stuff/DATA/Final Project/ietf-ambiguity-detection/corpus/processed/features/v3')
alignment_root = features_dir.parents[1] / 'alignments'
splits = ('train', 'validation', 'test')
rows_by_split = {}
for split in splits:
    rows = pd.read_csv(features_dir / 'rows' / f'{split}_rows.csv', dtype={'record_id': str, 'family_id': str, 'document_id': str, 'clause_ids': str})
    labels = pd.read_csv(features_dir / 'targets' / f'{split}_labels.csv', dtype={'record_id': str})
    rows_by_split[split] = rows.merge(labels, on='record_id', validate='one_to_one')
    assert len(rows_by_split[split]) == len(rows)

# Choose the frozen document input that best matches the row's clause IDs.
document_cache = {}
def load_ordered_clauses(family_id, document_id, wanted_ids):
    key = (family_id, document_id)
    if key not in document_cache:
        candidates = sorted((alignment_root / family_id).glob(f'*/inputs/{document_id}.json'))
        if not candidates:
            raise FileNotFoundError(f'No frozen clause input for {document_id}')
        best_clauses, best_score = None, -1
        for path in candidates:
            document = json.loads(path.read_text(encoding='utf-8'))
            clauses = document.get('clauses', [])
            score = len(wanted_ids & {str(clause.get('clause_id', '')) for clause in clauses})
            if score > best_score:
                best_clauses, best_score = clauses, score
        if not best_clauses or best_score <= 0:
            raise ValueError(f'No matching target clauses found for {document_id}')
        document_cache[key] = sorted(best_clauses, key=lambda clause: int(clause.get('index', 0)))
    return document_cache[key]

previous_by_split = {}
for split, frame in rows_by_split.items():
    previous_texts = []
    for row in frame[['family_id', 'document_id', 'clause_ids']].fillna('').to_dict(orient='records'):
        target_ids = {clause_id for clause_id in str(row['clause_ids']).split('|') if clause_id}
        ordered = load_ordered_clauses(str(row['family_id']), str(row['document_id']), target_ids)
        positions = [i for i, clause in enumerate(ordered) if clause.get('clause_id') in target_ids]
        if not positions:
            raise ValueError(f"Could not locate target {row['clause_ids']}")
        first, last = min(positions), max(positions)
        target_section = str(ordered[first].get('section_id', ''))
        previous = ''
        if first > 0 and str(ordered[first - 1].get('section_id', '')) == target_section:
            previous = str(ordered[first - 1].get('raw_text', ''))
        previous_texts.append(previous)
    previous_by_split[split] = previous_texts
    assert len(previous_texts) == len(frame)

print('Previous-clause context coverage:', {split: sum(bool(text.strip()) for text in previous_texts) for split, previous_texts in previous_by_split.items()})

# Reuse the existing target-clause + engineered matrix; fit the added context
# vocabulary on train only and append its sparse features.
context_vectorizer = TfidfVectorizer(lowercase=True, strip_accents='unicode', ngram_range=(1, 2), min_df=2, max_features=20000, sublinear_tf=True, token_pattern=r'(?u)\b[A-Za-z0-9][A-Za-z0-9_-]*\b')
context_matrices = {'train': context_vectorizer.fit_transform(previous_by_split['train'])}
for split in ('validation', 'test'):
    context_matrices[split] = context_vectorizer.transform(previous_by_split[split])
baseline_matrices = {split: sparse.load_npz(features_dir / 'matrices' / f'{split}_combined.npz') for split in splits}
augmented_matrices = {split: hstack([baseline_matrices[split], context_matrices[split]], format='csr') for split in splits}
y_by_split = {split: frame['label'].astype(int).to_numpy() for split, frame in rows_by_split.items()}

context_results = []
for name, matrices in (('baseline', baseline_matrices), ('with_previous_clause', augmented_matrices)):
    model = LogisticRegression(max_iter=2000, class_weight='balanced')
    model.fit(matrices['train'], y_by_split['train'])
    probability = {split: model.predict_proba(matrices[split])[:, 1] for split in splits}
    candidates = np.unique(probability['validation'])
    validation_f1 = [precision_recall_fscore_support(y_by_split['validation'], probability['validation'] >= threshold, average='binary', zero_division=0)[2] for threshold in candidates]
    selected_threshold = float(max(zip(candidates, validation_f1), key=lambda item: (item[1], item[0]))[0])
    for split in ('validation', 'test'):
        for threshold_name, threshold in (('default_0.50', 0.50), ('validation_best_F1', selected_threshold)):
            prediction = probability[split] >= threshold
            precision, recall, f1, _ = precision_recall_fscore_support(y_by_split[split], prediction, average='binary', zero_division=0)
            context_results.append({'model': name, 'split': split, 'threshold': threshold_name, 'threshold_value': threshold,
                                    'AP': average_precision_score(y_by_split[split], probability[split]),
                                    'precision': precision, 'recall': recall, 'F1': f1,
                                    'CM_[TN,FP,FN,TP]': confusion_matrix(y_by_split[split], prediction, labels=[0, 1]).ravel().tolist()})
    print(f'{name}: validation-selected threshold={selected_threshold:.3f}')
print(pd.DataFrame(context_results).round(3).to_string(index=False))

Previous-clause context coverage: {'train': 2174, 'validation': 134, 'test': 136}
baseline: validation-selected threshold=0.422
with_previous_clause: validation-selected threshold=0.374
               model      split          threshold  threshold_value    AP  precision  recall    F1 CM_[TN,FP,FN,TP]
            baseline validation       default_0.50            0.500 0.283      0.250   0.385 0.303 [83, 30, 16, 10]
            baseline validation validation_best_F1            0.422 0.283      0.283   0.654 0.395  [70, 43, 9, 17]
            baseline       test       default_0.50            0.500 0.315      0.305   0.529 0.387 [79, 41, 16, 18]
            baseline       test validation_best_F1            0.422 0.315      0.300   0.706 0.421 [64, 56, 10, 24]
with_previous_clause validation       default_0.50            0.500 0.291      0.237   0.346 0.281  [84, 29, 17, 9]
with_previous_clause validation validation_best_F1            0.374 0.291      0.247   0.692 0.364  [58, 55, 8, 18]
wi

### Context experiment result

Previous-clause text was available for 2,174/2,253 train, 134/139 validation, and 136/154 test rows. On the existing split, AP changed from 0.315 to 0.316; with validation-selected thresholds, test F1 changed from 0.421 to 0.400. This experiment did not show a clear improvement.

In family-held-out evaluation with the two small families retained in training, baseline versus context AP was 0.291 versus 0.292, while F1 at 0.50 was 0.365 versus 0.325. These results are exploratory.

## 6. Earlier revision-history feature experiment

A previous exploratory run added features comparing a clause with its preceding draft revision. It reported validation AP 0.332, test AP 0.306, and test F1 0.230 at validation-selected threshold 0.53. This heuristic did not improve the tested result and is recorded here for completeness, not as a selected model.